In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader

# **1. Datenvorbereitung**
transform = transforms.Compose([
    transforms.Resize((128, 128)),  # Bilder auf gleiche Größe skalieren
    transforms.ToTensor(),          # In Tensor umwandeln
    transforms.Normalize((0.5,), (0.5,))  # Normalisierung
])

# Beispiel-Datensatz (z. B. für Testzwecke)
train_dataset = datasets.FakeData(transform=transform)  # Hier echten Datensatz einfügen
test_dataset = datasets.FakeData(transform=transform)

# DataLoader für Batches
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# **2. CNN-Modell definieren**
class CNNModel(nn.Module):
    def __init__(self, num_classes=5):  # 5 Klassen für das Rating
        super(CNNModel, self).__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 16 * 16, 256),  # Inputgröße abhängig von Bildgröße
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)  # Ausgabe entspricht Anzahl der Klassen
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = self.fc_layers(x)
        return x

# **3. Modell, Loss & Optimizer definieren**
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CNNModel(num_classes=5).to(device)
criterion = nn.CrossEntropyLoss()  # Klassifikationsverlust
optimizer = optim.Adam(model.parameters(), lr=0.001)

# **4. Training & Validierung**
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}")

# **5. Modell testen**
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Test Accuracy: {100 * correct / total:.2f}%")
